<a href="https://colab.research.google.com/github/DulceMarinG/Compress-Image-python/blob/main/Copy_of_Hands_On_Prompt_Engineering_y_Sistemas_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: PROMPT ENGINEERING Y SISTEMAS RAG**

Una vez vista la masterclass ***Prompt Engineering y Sistemas RAG***, se proporciona el siguiente ***Colab*** para construir, en vivo, distintas estrategias de prompting y un mini sistema de RAG.

Usamos **Groq** para tener acceso a inferencia con GPU gratuita, sin necesidad de infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1EyWl33ZyAWuyMKXz_mr9aaNu3JF8e8vS?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Para no exponer tu ***API key*** directamente en el código, Colab ofrece un panel de ***Secrets*** (ícono de llave en la barra lateral izquierda) donde puedes guardarla de forma segura, añadiendo un nombre asociado a la ***API key*** para guardarla dentro de una variable y usarla dentro del notebook.

In [1]:
# Instalar cliente de Groq y leer API key desde Colab Secrets
!pip install Groq --quiet

from groq import Groq
from google.colab import userdata

client = Groq(api_key=userdata.get('API_KEY_GROQ'))
print("Cliente de Groq inicializado correctamente.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 4.7 MB/s eta 0:00:00
Cliente de Groq inicializado correctamente.


## **ESTRATEGIAS DE PROMPTS**

### **ZERO-SHOT VS. FEW-SHOT**

Zero-shot es pedirle al modelo que haga algo sin darle ejemplos. Few-shot le muestra dos o tres ejemplos de entrada y salida antes de pedirle la tarea real. Vamos a comparar ambas estrategias con la misma tarea de clasificación.

In [2]:
# Prompt de clasificación en modo zero-shot
prompt_zero_shot = "Clasifica el sentimiento de esta reseña en Positivo, Negativo o Mixto: 'El envío llegó tarde pero el producto es excelente.' Respuesta muy breve y corta."

response_zero = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_zero_shot}]
)

print("Zero-shot:", response_zero.choices[0].message.content)


Zero-shot: Mixto


In [3]:
# Prompt de clasificación en modo few-shott
prompt_few_shot = """Clasifica el sentimiento de cada reseña como Positivo, Negativo o Mixto.

Reseña: "Me encantó, llegó rápido y en perfecto estado."
Sentimiento: Positivo

Reseña: "Nunca llegó mi pedido, pésimo servicio."
Sentimiento: Negativo

Reseña: "El envío llegó tarde pero el producto es excelente."
Sentimiento:"""

response_few = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_few_shot}]
)

print("Few-shot:", response_few.choices[0].message.content)


Few-shot: Sentimiento: Mixto


### **CHAIN-OF-THOUGHT**

Chain-of-thought le pide al modelo mostrar su razonamiento paso a paso antes de la respuesta final, algo que mejora notablemente el desempeño en problemas de lógica.

In [4]:
# Razonamiento paso a paso (chain-of-thought)
problema = (
    "Un tren sale de la ciudad A a 80 km/h. Dos horas después, otro tren sale de la misma "
    "ciudad hacia el mismo destino a 120 km/h. ¿Cuánto tiempo tarda el segundo tren en "
    "alcanzar al primero? Muestra tu razonamiento paso a paso antes de dar la respuesta final."
)

response_cot = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": problema}]
)

print(response_cot.choices[0].message.content)
# problema_directo = problema.split(".")[-2]  # variante: pedir solo el número, sin razonamiento, y comparar qué tan seguido se equivoca


**Paso a paso**

1. **Identificar el desfase inicial**  
   - El primer tren parte de la ciudad **A** a \(80\ \text{km/h}\).  
   - El segundo tren parte **dos horas después** con una velocidad de \(120\ \text{km/h}\).  
   - Mientras el segundo tren aún no ha salido, el primero ya ha avanzado:  
     \[
     \text{Distancia inicial} = 80\ \text{km/h} \times 2\ \text{h} = 160\ \text{km}.
     \]
     Por lo tanto, cuando el segundo tren parte, el primero está 160 km por delante.

2. **Expresar las distancias en función del tiempo**  
   Sea \(t\) el tiempo (en horas) transcurrido **después** de que el segundo tren haya partido.

   - **Distancia del primer tren** (ya había recorrido 160 km y sigue moviéndose a 80 km/h):  
     \[
     d_1(t) = 160 + 80\,t.
     \]
   - **Distancia del segundo tren** (parte a 120 km/h desde el punto de partida):  
     \[
     d_2(t) = 120\,t.
     \]

3. **Condición de captura**  
   El segundo tren alcanza al primero cuando sus distancias desde la ciu

### **EL LÍMITE DEL PROMPT: LO QUE EL MODELO NUNCA VIO**

Ninguna técnica de prompting le da información nueva al modelo. Si le preguntamos algo que no pudo haber visto en su entrenamiento, puede alucinar una respuesta que suene convincente pero no sea real.

In [6]:
# Preguntar algo que el modelo no pudo haber visto en su entrenamiento y observar si alucina
prompt_desconocido = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)
# Por más segura que suene la respuesta, el modelo no tiene forma de saber esto: es una alucinación



Lo siento, no dispongo de esa información.


## **RAG: BUSCAR ANTES DE RESPONDER**

RAG separa el proceso en dos pasos: primero un sistema de búsqueda encuentra los fragmentos más relevantes de una base de conocimiento propia (usando *embeddings* y similitud semántica); después, esos fragmentos se le entregan al modelo junto con la pregunta para generar la respuesta.

In [7]:
# Instalar sentence-transformers
!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np


In [9]:
# Definir la base de conocimiento (política de devoluciones) y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "Las devoluciones se aceptan hasta 30 días después de la compra, con el producto en su "
    "empaque original.",
    "Los envíos internacionales no tienen devolución gratuita; el cliente cubre el costo de "
    "envío de regreso.",
    "Los productos en oferta o liquidación no son elegibles para devolución, solo para "
    "cambio de talla."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


In [10]:
# Definir una función que calcule la similitud entre la pregunta y cada fragmento, y regrese el más relevante
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo devolver algo que compré en oferta?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)


Fragmento recuperado: Los productos en oferta o liquidación no son elegibles para devolución, solo para cambio de talla.


In [11]:
# Enviar la pregunta junto con el fragmento recuperado al modelo y mostrar la respuesta con RAG
prompt_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

response_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_rag}]
)

print(response_rag.choices[0].message.content)
# Compara esta respuesta contra lo que el modelo diría sin el fragmento: sin RAG probablemente improvisaría una política genérica


No, los productos en oferta no son elegibles para devolución, solo pueden cambiarse de talla.


# **CHALLENGE: ASISTENTE DE POLÍTICAS CON RAG**

Una vez visto el ***Hands-On: Prompt Engineering y Sistemas RAG***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño asistente con RAG sobre un documento propio, comparando la respuesta **con RAG** contra la respuesta **sin RAG** para la misma pregunta. En esta solución se usa como base de conocimiento el propio reglamento de evaluación del curso IA Aplicada con Llama.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

## **INSTRUCCIONES:**

**1. Define tu base de conocimiento y genera embeddings:**

* Lee la API key previamente configurada desde **Colab Secrets** e instala/importa las librerías necesarias.

* Construye una lista llamada `documentos` con 3 fragmentos de un documento real de tu propio contexto (reglamento, políticas, FAQs, etc.) y genera sus embeddings con `sentence-transformers`.

In [12]:
# Leer API key, instalar e importar librerías
!pip install Groq --quiet

from groq import Groq
from google.colab import userdata

!pip install sentence-transformers --quiet

from sentence_transformers import SentenceTransformer
import numpy as np

client = Groq(api_key=userdata.get('API_KEY_GROQ'))
print("Cliente de Groq inicializado correctamente.")



Cliente de Groq inicializado correctamente.


In [24]:
# Definir la lista documentos y generar sus embeddings
modelo_embeddings = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

documentos = [
    "El periodo límite para solicitar el cambio o devolución suele variar entre 5, 30 o 60 días naturales desde la entrega o compra "
    "con su ticket de compra.",
    "El artículo debe estar sin usar, en su empaque original, con etiquetas, sellos intactos y con todos sus accesorios o manuales. "
    "para hacer efectiva su devolución.",
    " Artículos que no aplican para devolución son ropa interior, productos perecederos, artículos en venta final o personalizados "
    "sin excepción."
]

embeddings_documentos = modelo_embeddings.encode(documentos)
print("Embeddings generados:", embeddings_documentos.shape)



Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings generados: (3, 384)


**2. Recupera el fragmento relevante:** Define una función `buscar_fragmento(pregunta)` que calcule la similitud coseno y regrese el fragmento más relevante para una pregunta dada.

In [29]:
# Definir la función buscar_fragmento
def buscar_fragmento(pregunta):
    embedding_pregunta = modelo_embeddings.encode([pregunta])
    similitudes = np.dot(embeddings_documentos, embedding_pregunta.T).flatten()
    indice_mas_similar = np.argmax(similitudes)
    return documentos[indice_mas_similar]

pregunta = "¿Puedo regresar algo que compre, sin ticket?"
fragmento = buscar_fragmento(pregunta)
print("Fragmento recuperado:", fragmento)


Fragmento recuperado: El periodo límite para solicitar el cambio o devolución suele variar entre 5, 30 o 60 días naturales desde la entrega o compra con su ticket de compra.


**3. Genera la respuesta sin RAG:** Envía una pregunta real sobre tu documento directamente al modelo (sin ningún fragmento de contexto) y guarda la respuesta en `respuesta_sin_rag`.

In [33]:
prompt_sin_rag = f"""Responde la pregunta del cliente. Si no sabes la respuesta, dilo claramente.

Pregunta: {pregunta}

Respuesta muy breve y corta."""

respuesta_sin_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_sin_rag}]
)

print(respuesta_sin_rag.choices[0].message.content)

Sí, pero revisa la política de la tienda.


**4. Genera la respuesta con RAG:** Recupera el fragmento relevante con tu función y envía la pregunta junto con ese fragmento al modelo. Guarda la respuesta en `respuesta_con_rag`.

In [31]:
# Consultar la pregunta con RAG y guardar el resultado en respuesta_con_rag
prompt_con_rag = f"""Responde la pregunta del cliente usando SOLO la siguiente política de la tienda. Si la política no cubre la pregunta, dilo claramente.

Política: {fragmento}

Pregunta: {pregunta}

Respuesta muy breve y corta."""

respuesta_con_rag = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_con_rag}]
)

print(respuesta_con_rag.choices[0].message.content)

No, sin ticket no se puede realizar la devolución.


**5. Compara y concluye:** Imprime ambas respuestas y concluye cuál de las dos evitó mejor una alucinación o dio una respuesta más precisa.

In [35]:
# Mostrar ambas respuestas para comparar
print("Respuesta sin RAG:", respuesta_sin_rag.choices[0].message.content)
print("Respuesta con RAG:", respuesta_con_rag.choices[0].message.content)

prompt_desconocido = (
    f"""Responde la pregunta del cliente. Si no sabes la respuesta, dilo claramente.

Pregunta: {pregunta}

Respuesta muy breve y corta."""

)

response_alucinacion = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

prompt_desconocido1 = (
    "¿Cuál fue el resultado de la final del hackathon interno de DEV.F del 14 de agosto de "
    "2026? Respuesta muy breve y corta."
)

response_alucinacion1 = client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt_desconocido}]
)

print(response_alucinacion.choices[0].message.content)

Respuesta sin RAG: Sí, pero revisa la política de la tienda.
Respuesta con RAG: No, sin ticket no se puede realizar la devolución.
No. La devolución sin ticket normalmente no se acepta.
